In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim

--2026-07-03 15:36:12--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2026-07-03 15:36:12 (140 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab  # This will now correctly be 50257

# Cell 2: Re-encode the data so all token IDs fit within 0 to 50256
data = torch.tensor(enc.encode(text), dtype=torch.long)

# Cell 3: Split your data
n = int(0.9 * data.shape[0])
train_data = data[:n]
val_data = data[n:]

In [20]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# ==========================================
# HYPERPARAMETERS (Scaled to ~6M Params)
# ==========================================
# Hyperparameters
block_size = 128
batch_size = 64

max_iters = 20000

learning_rate = 3e-4

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1
eval_interval = 200
# vocab_size = enc.n_vocab # Make sure this is defined in your environment

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==========================================
# DATA LOADING
# ==========================================
# Assuming train_data and val_data are already defined
train_data = train_data.to(device)
val_data = val_data.to(device)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,), device=device)

    start_offsets = ix.unsqueeze(1)
    sequence_offsets = torch.arange(block_size, device=device)

    x = data[start_offsets + sequence_offsets]
    y = data[start_offsets + sequence_offsets + 1]
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters, device=device)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ==========================================
# MODEL CLASSES
# ==========================================
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

    def forward(self, x):
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # Native PyTorch FlashAttention handles the heavy lifting
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True
        )
        return out

class Multihead(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.head = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.head], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = Multihead(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying
        self.lm_head.weight = self.token_embedding_table.weight

        # Initialize weights
        self.apply(self._init_weights)

        # Scale residual projections (GPT-2)
        for pn, p in self.named_parameters():
            if pn.endswith("proj.weight"):
                torch.nn.init.normal_(
                    p,
                    mean=0.0,
                    std=0.02 / (2 * n_layer) ** 0.5,
                )

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx
model = BigramLanguageModel().to(device)

print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=0.1
)
# ==========================================
# TRAINING LOOP
# ==========================================

7.24 M parameters


In [21]:
# ==========================================
# TRAINING LOOP
# ==========================================
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')

    # Standard forward/backward pass (no AMP)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(enc.decode(model.generate(context, max_new_tokens=20)[0].tolist()))

step 0: train loss 10.8337, val loss 10.8408
step 200: train loss 5.6264, val loss 5.8183
step 400: train loss 4.8068, val loss 5.2404
step 600: train loss 4.4532, val loss 5.0765
step 800: train loss 4.2197, val loss 4.9491
step 1000: train loss 4.0561, val loss 4.8910
step 1200: train loss 3.9134, val loss 4.8966
step 1400: train loss 3.7838, val loss 4.9046
step 1600: train loss 3.6666, val loss 4.9117
step 1800: train loss 3.5552, val loss 4.9616


KeyboardInterrupt: 

In [25]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(enc.decode(model.generate(context, max_new_tokens=4000)[0].tolist()))

! 'twas you that you shall hold, since
So ran well and the city!

Roman:
Must he knows with baseness.

Allvenient were said, sir, hisself my money fellow.

FLORIZEL:
Indeed, but for 'tis very virtue of my
When can be better along with him.VALERIA:
You may, your voices, shall be better.
there doth not stir against theShericians: I I have said, for
will be o' the matter; the cause give the knaves; he
son hear me, let's be so.

LELBOW:
I hate, there; whom, the time dNever, in that may be omit
 fray awhile: the Nature I can have harm and
For timeily I am both of love.' cockck thy death
Is a temporcare; but a sacred idle and now
ounds the outwardators of love as they may wrong,
And I live or else become so.
What is the friar?

DUKE OF AUMER:
What sin's in blood of heaven?
I wast done, sir? yet she were it here!
my head. blinked torches that here't please thee,
Pona, and when I rue into a man mean:
Unless hath feet beetle sense an fortunes forth proffer'd,
Did dotardel with pride, to make th